# Agent Running Test 
### Prerequisite for LLM used in the agent example
1. Install Ollama from https://ollama.com
2. Start Ollama `ollama run qwen3:4b`


### Prerequisites for a remote VM
1. Start an SSH server (e.g., `openssh-server`) so you can connect to the VM if needed.
2. Install Python virtual environment support (e.g., `python3-venv`).


In [1]:
import torch

if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected - PyTorch is using CPU-only version or no CUDA drivers")

GPU device: Quadro RTX 8000


In [2]:
import os

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import transformers

transformers.logging.set_verbosity_error()

In [3]:
example_path = 'agent.py'

## Python Interpreter

In [ ]:
!python {example_path}

## Run the agent and tool teogether locally

In [ ]:
# Removed pip_install parameter from decorators in {example_path}
import ast 
import os
import tempfile


with open(example_path, "r") as f:
    source_code = f.read()
    mod = ast.parse(source_code)

# Remove env keyword argument
for node in ast.walk(mod):
    if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
        for dec in node.decorator_list:
            if isinstance(dec, ast.Call):
                dec.keywords = [kw for kw in dec.keywords if kw.arg != "pip_install"]

with tempfile.NamedTemporaryFile("w", delete=False, suffix=".py") as temp_file:
    temp_file.write(ast.unparse(mod))
    temp_file_path = temp_file.name

!single-spec run {temp_file_path}
os.unlink(temp_file_path)

Tool 'Grounding' assigned to Node 'local' 
The grounding tool detected two bounding boxes for the target entity "cat" in the image. Here are the coordinates:

1. **First Box**: Top-left (9, 54) to Bottom-right (316, 474)
2. **Second Box**: Top-left (346, 23) to Bottom-right (639, 372)

These boxes represent the detected positions of cats in the image with a confidence threshold of 0.3.


## Run the agent locally while executing the tool inside a local Python virtual environment.

In [ ]:
!single-spec run {example_path}

^C


## Run the agent locally while executing the tool inside a remote Python virtual environment.

In [ ]:
!single-spec run {example_path} -c "exec_nodes_remote.toml"

Tool 'Grounding' assigned to Node 'vm1' 

STARTING REMOTE DEPLOYMENT: Grounding
Connecting to 172.16.51.51:22 via SSH...
✓ Connected to 172.16.51.51:22 via SSH
Allocating remote port...
✓ Allocated remote port 41153 on 172.16.51.51
Writing code and preparing requirements...
Creating remote directory /tmp/.single_spec_cache/Grounding...
Uploading files via SFTP...
✓ Copied code to remote directory /tmp/.single_spec_cache/Grounding
Creating virtual environment...
✓ Created venv at /tmp/.single_spec_cache/Grounding/.venv
Installing 1 special package(s)...
  Installing torch from https://download.pytorch.org/whl/cu124...
    ✓ Installed torch
Installing packages from requirements.txt...
✓ Installed all requirements
Starting server...
  Waiting for server to start...
  Testing connectivity from local machine to http://172.16.51.51:41153...
  ✓ Server is accessible from local machine at http://172.16.51.51:41153
The grounding tool detected two bounding boxes for the target entity "cat" in th

## Clear Cache on nodes

In [ ]:
!single-spec clear